# Week 6: Task 3 - Housing Price Prediction (Advanced Models & Evaluation)

**Internship:** Arch Technologies, Machine Learning Domain, Month 2  
**Project:** California Housing Price Prediction (Second Half: Advanced Ensembles, Tuning & Final Evaluation)  

## Weekly Breakdown & Objectives:
- **Monday (Advanced Model Training):** Load preprocessed arrays from `week-5/outputs/` and train Random Forest & Gradient Boosting Regressors.
- **Tuesday (Model Evaluation & Comparison):** Evaluate test set RMSE, MAE, and R² against the Week 5 Linear Regression baseline; generate comparison bar chart.
- **Wednesday (Hyperparameter Tuning):** Tune the best model (Random Forest) via `GridSearchCV` with cross-validation.
- **Thursday (Final Model Testing):** Inspect sample test instance predictions, errors, and generate predicted vs. actual scatter plot.
- **Friday (Final Summary & Documentation):** Synthesize model metrics, overfitting resolution, and conclusions for Task 3.

## 1. Environment & Path Setup
We import required libraries and configure robust relative path resolution pointing to `week-5/outputs/` and `week-6/outputs/`.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline

# Resolve repository paths dynamically
CURRENT_DIR = Path.cwd()
if CURRENT_DIR.name == 'notebooks':
    REPO_ROOT = CURRENT_DIR.parents[1]
elif CURRENT_DIR.name == 'week-6':
    REPO_ROOT = CURRENT_DIR.parent
else:
    REPO_ROOT = CURRENT_DIR

WEEK5_OUTPUTS = REPO_ROOT / 'week-5' / 'outputs'
WEEK6_OUTPUTS = REPO_ROOT / 'week-6' / 'outputs'
print('Environment and paths configured successfully.')

Environment and paths configured successfully.


## 2. Load Preprocessed Data (Week 5 Outputs)
Loading the preprocessed training and testing arrays (`X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv`). Features were engineered, missing values imputed, and numerical columns standardized during Week 5.

In [2]:
X_train = pd.read_csv(WEEK5_OUTPUTS / 'X_train.csv')
X_test = pd.read_csv(WEEK5_OUTPUTS / 'X_test.csv')
y_train = pd.read_csv(WEEK5_OUTPUTS / 'y_train.csv').squeeze('columns')
y_test = pd.read_csv(WEEK5_OUTPUTS / 'y_test.csv').squeeze('columns')

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'y_test shape:  {y_test.shape}')
print(f'Features: {list(X_train.columns)}')

X_train shape: (16500, 13)
X_test shape:  (4126, 13)
y_train shape: (16500,)
y_test shape:  (4126,)
Features: ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'rooms_per_household', 'bedrooms_per_room', 'population_per_household', 'ocean_proximity_INLAND', 'ocean_proximity_ISLAND']


## 3. Monday: Advanced Model Training
We train two non-linear ensemble models using reasonable default hyperparameters:
1. **Random Forest Regressor:** Bagging ensemble of 100 decision trees.
2. **Gradient Boosting Regressor:** Boosting ensemble of 100 decision trees trained sequentially.

In [3]:
rf_path = WEEK6_OUTPUTS / 'random_forest_model.pkl'
gb_path = WEEK6_OUTPUTS / 'gradient_boosting_model.pkl'

if rf_path.exists() and gb_path.exists():
    print('Loading trained models from week-6/outputs/...')
    rf_model = joblib.load(rf_path)
    gb_model = joblib.load(gb_path)
else:
    print('Training models...')
    rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train, y_train)
    gb_model = GradientBoostingRegressor(n_estimators=100, random_state=42)
    gb_model.fit(X_train, y_train)

print(f'Random Forest Regressor: {rf_model}')
print(f'Gradient Boosting Regressor: {gb_model}')

Loading trained models from week-6/outputs/...
Random Forest Regressor: RandomForestRegressor(n_jobs=-1, random_state=42)
Gradient Boosting Regressor: GradientBoostingRegressor(random_state=42)


## 4. Tuesday: Model Evaluation & Comparison against Baseline
We evaluate test set predictions across **Root Mean Squared Error (RMSE)**, **Mean Absolute Error (MAE)**, and **R² Score**, and benchmark them against the Week 5 Linear Regression baseline (`week-5/outputs/baseline_results.txt`).

In [4]:
def evaluate(y_true, y_pred):
    return {
        'RMSE ($)': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'MAE ($)': float(mean_absolute_error(y_true, y_pred)),
        'R2 Score': float(r2_score(y_true, y_pred))
    }

rf_preds = rf_model.predict(X_test)
gb_preds = gb_model.predict(X_test)

baseline_metrics = {'RMSE ($)': 71750.18, 'MAE ($)': 52006.98, 'R2 Score': 0.6353}
comparison_df = pd.DataFrame({
    'Linear Regression (Baseline)': baseline_metrics,
    'Gradient Boosting Regressor': evaluate(y_test, gb_preds),
    'Random Forest Regressor (Default)': evaluate(y_test, rf_preds)
}).T

print('Model Comparison Table:')
print('-' * 80)
print(f'{"Model":33} | {"RMSE ($)":>13} | {"MAE ($)":>13} | {"R2 Score":>8}')
print('-' * 80)
for model_name, row in comparison_df.iterrows():
    print(f'{model_name:33} | ${row["RMSE ($)"]:>12,.2f} | ${row["MAE ($)"]:>12,.2f} | {row["R2 Score"]:8.4f}')
print('-' * 80)

Model Comparison Table:
--------------------------------------------------------------------------------
Model                             |      RMSE ($) |       MAE ($) | R2 Score
--------------------------------------------------------------------------------
Linear Regression (Baseline)      |    $71,750.18 |    $52,006.98 |   0.6353
Gradient Boosting Regressor       |    $57,393.75 |    $39,705.75 |   0.7616
Random Forest Regressor (Default) |    $49,780.29 |    $32,150.44 |   0.8207
--------------------------------------------------------------------------------


In [5]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
models = list(comparison_df.index)
colors = ['#4A90E2', '#F5A623', '#50E3C2']

# 1. RMSE
axes[0].bar(models, comparison_df['RMSE ($)'], color=colors, edgecolor='black', alpha=0.85)
axes[0].set_title('Root Mean Squared Error (RMSE)\n(Lower is Better)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('RMSE ($)', fontsize=11)
axes[0].tick_params(axis='x', rotation=18)
for i, v in enumerate(comparison_df['RMSE ($)']):
    axes[0].text(i, v + 1200, f'${v:,.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# 2. MAE
axes[1].bar(models, comparison_df['MAE ($)'], color=colors, edgecolor='black', alpha=0.85)
axes[1].set_title('Mean Absolute Error (MAE)\n(Lower is Better)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('MAE ($)', fontsize=11)
axes[1].tick_params(axis='x', rotation=18)
for i, v in enumerate(comparison_df['MAE ($)']):
    axes[1].text(i, v + 900, f'${v:,.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# 3. R2
axes[2].bar(models, comparison_df['R2 Score'], color=colors, edgecolor='black', alpha=0.85)
axes[2].set_title('R² Score (Explained Variance)\n(Higher is Better)', fontsize=12, fontweight='bold')
axes[2].set_ylabel('R² Score', fontsize=11)
axes[2].set_ylim(0, 1.0)
axes[2].tick_params(axis='x', rotation=18)
for i, v in enumerate(comparison_df['R2 Score']):
    axes[2].text(i, v + 0.02, f'{v:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle('Model Performance Comparison - Tuesday Deliverable', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(WEEK6_OUTPUTS / 'model_comparison_chart.png', dpi=300, bbox_inches='tight')
print('Comparison chart saved to week-6/outputs/model_comparison_chart.png')
plt.show()

Comparison chart saved to week-6/outputs/model_comparison_chart.png


## 5. Wednesday: Hyperparameter Tuning
Because **Random Forest Regressor** demonstrated the strongest performance (RMSE of \$49,780 vs \$57,393 for Gradient Boosting), we optimize it via `GridSearchCV` with 3-fold cross-validation.

In [6]:
tuned_path = WEEK6_OUTPUTS / 'housing_rf_model_tuned.pkl'
if tuned_path.exists():
    print(f'Loading tuned Random Forest from {tuned_path.name}...')
    tuned_rf = joblib.load(tuned_path)
else:
    param_grid = {'n_estimators': [100, 150], 'max_depth': [20, None], 'min_samples_split': [2, 5]}
    grid = GridSearchCV(RandomForestRegressor(random_state=42), param_grid, cv=3, scoring='neg_root_mean_squared_error', n_jobs=-1)
    grid.fit(X_train, y_train)
    tuned_rf = grid.best_estimator_

tuned_preds = tuned_rf.predict(X_test)
tuned_metrics = evaluate(y_test, tuned_preds)
default_metrics = evaluate(y_test, rf_preds)

print('Tuned Model Performance:')
print(f"  RMSE:     ${tuned_metrics['RMSE ($)']:,.2f}")
print(f"  MAE:      ${tuned_metrics['MAE ($)']:,.2f}")
print(f"  R2 Score: {tuned_metrics['R2 Score']:.4f}")
print('Improvement over Default RF:')
print(f"  RMSE: +{((default_metrics['RMSE ($)'] - tuned_metrics['RMSE ($)']) / default_metrics['RMSE ($)'] * 100):.2f}% reduction")
print(f"  MAE:  +{((default_metrics['MAE ($)'] - tuned_metrics['MAE ($)']) / default_metrics['MAE ($)'] * 100):.2f}% reduction")

Loading tuned Random Forest from week-6/outputs/housing_rf_model_tuned.pkl...
Tuned Model Performance:
  RMSE:     $49,666.10
  MAE:      $32,027.45
  R2 Score: 0.8215
Improvement over Default RF:
  RMSE: +0.23% reduction
  MAE:  +0.38% reduction


## 6. Thursday: Final Model Testing & Visual Diagnostics
We evaluate the tuned model on 10 sample test instances, calculate absolute and percentage errors, and plot actual vs. predicted values for the full test set.

In [7]:
np.random.seed(42)
sample_idx = np.random.choice(len(y_test), size=10, replace=False)
sample_idx.sort()

actuals = y_test.iloc[sample_idx].values
preds = tuned_preds[sample_idx]
diffs = np.abs(actuals - preds)
pcts = (diffs / actuals) * 100.0

print('Sample Predictions (10 Diverse Held-out Test Instances):')
print('-' * 80)
print(f'{"Index":5} | {"Actual Value":>16} | {"Predicted Value":>16} | {"Absolute Error":>16} | {"Percentage Error":>16}')
print('-' * 80)
for i in range(len(sample_idx)):
    print(f'{sample_idx[i]:5d} | ${actuals[i]:>15,.2f} | ${preds[i]:>15,.2f} | ${diffs[i]:>15,.2f} | {pcts[i]:15.2f}%')
print('-' * 80)
print(f'Average Sample Absolute Error:   ${diffs.mean():,.2f}')
print(f'Average Sample Percentage Error: {pcts.mean():.2f}%')

Sample Predictions (10 Diverse Held-out Test Instances):
--------------------------------------------------------------------------------
Index |     Actual Value |  Predicted Value |   Absolute Error | Percentage Error
--------------------------------------------------------------------------------
  864 |      $159,800.00 |      $146,908.67 |       $12,891.33 |            8.07%
 1042 |      $295,200.00 |      $374,998.05 |       $79,798.05 |           27.03%
 1104 |      $329,600.00 |      $330,044.00 |          $444.00 |            0.13%
 1507 |      $500,001.00 |      $499,888.33 |          $112.67 |            0.02%
 1766 |      $155,000.00 |       $92,390.00 |       $62,610.00 |           40.39%
 1948 |      $102,800.00 |      $144,105.33 |       $41,305.33 |           40.18%
 2867 |      $128,100.00 |      $134,111.33 |        $6,011.33 |            4.69%
 2925 |      $210,100.00 |      $159,112.67 |       $50,987.33 |           24.27%
 3339 |      $366,700.00 |      $376,078.03

In [8]:
fig, ax = plt.subplots(figsize=(9, 7))
residuals = np.abs(y_test.values - tuned_preds)

scatter = ax.scatter(
    y_test.values,
    tuned_preds,
    c=residuals,
    cmap='viridis',
    alpha=0.45,
    s=25,
    label='Held-out Test Blocks'
)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Absolute Error ($)', fontsize=11)

min_val = min(y_test.min(), tuned_preds.min())
max_val = max(y_test.max(), tuned_preds.max())
ax.plot([min_val, max_val], [min_val, max_val], color='crimson', linestyle='--', linewidth=2, label='Perfect Prediction (y = x)')

# Highlight 10 sample instances
ax.scatter(actuals, preds, color='red', s=90, marker='o', edgecolor='black', linewidth=1.5, label='Sample Instances', zorder=5)

ax.set_title('Predicted vs. Actual Housing Prices (Tuned Random Forest)', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Actual Median House Value ($)', fontsize=11)
ax.set_ylabel('Predicted Median House Value ($)', fontsize=11)
ax.legend(loc='upper left', frameon=True)
ax.set_xlim(min_val - 10000, max_val + 10000)
ax.set_ylim(min_val - 10000, max_val + 10000)

plt.tight_layout()
plt.savefig(WEEK6_OUTPUTS / 'sample_predictions.png', dpi=300, bbox_inches='tight')
print('Scatter plot saved to week-6/outputs/sample_predictions.png')
plt.show()

Scatter plot saved to week-6/outputs/sample_predictions.png


## 7. Feature Importance Diagnostics
We inspect feature importances from the tuned Random Forest model to understand the driving signals for California housing prices.

In [9]:
importances = pd.Series(tuned_rf.feature_importances_, index=X_train.columns).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
importances.plot(kind='barh', color='#2E7D32', edgecolor='black', alpha=0.85)
plt.title('Feature Importances - Tuned Random Forest Regressor', fontsize=13, fontweight='bold')
plt.xlabel('Gini Importance', fontsize=11)
plt.ylabel('Feature', fontsize=11)
plt.tight_layout()
plt.show()

## 8. Friday: Finalization & Conclusions

### Summary of Findings:
1. **Ensemble Superiority:** Tuned Random Forest achieved an **R² score of 0.8215** and **RMSE of \$49,666.10**, outperforming both Linear Regression (\$71,750.18) and Gradient Boosting (\$57,393.75).
2. **Baseline Comparison:** The tuned model achieved a **30.78% reduction in RMSE** and **38.42% reduction in MAE** over the Week 5 baseline.
3. **Predictive Drivers:** Median income, geographical proximity (ocean proximity), and ratio features (rooms per household, bedrooms per room) were the strongest predictors.
4. **Task 3 Status:** Complete.

**Next Task:** **Task 4 (Iris Flower Classification)** begins in Week 7.